# Taller 04 — K-Means con datos sintéticos y métodos de verificación

Implementación de **K-Means** sobre un conjunto sintético bidimensional con **seis grupos gaussianos isotrópicos** de distinta densidad, **dos grupos parcialmente superpuestos**, **outliers uniformes** y selección del número óptimo de clusters entre **K = 2 … 12** mediante:

1. Coeficiente de **Silhouette**
2. Método del **codo** (inercia)
3. Métricas con **ground truth** (ARI)

Referencia: cuaderno de clustering K-Means del repositorio del curso.

## 1. Importaciones

In [ ]:
try:
    import numpy as np
except ImportError:
    %pip install numpy
    import numpy as np

try:
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
except ImportError:
    %pip install matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm

try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import (
        adjusted_rand_score,
        silhouette_samples,
        silhouette_score,
    )
    from sklearn.neighbors import LocalOutlierFactor
except ImportError:
    %pip install scikit-learn
    from sklearn.cluster import KMeans
    from sklearn.metrics import (
        adjusted_rand_score,
        silhouette_samples,
        silhouette_score,
    )
    from sklearn.neighbors import LocalOutlierFactor

## 2. Generación de datos sintéticos

Se construyen **6 grupos** con las siguientes propiedades:

- Distribución **gaussiana isotrópica** (`std` igual en X e Y).
- **Mismo número de puntos** por grupo, pero **distinta desviación estándar** (distinta densidad).
- Dos grupos con centros **cercanos** para lograr **superposición parcial**.
- Todos los inliers se generan de forma que, al visualizar el conjunto completo, queden en el rango **[-10, 10]** en ambos ejes.
- Se agregan **outliers** con distribución **uniforme** en `[-10, 10] × [-10, 10]`.

Las etiquetas `-1` identifican outliers; las etiquetas `0 … 5` identifican los seis grupos verdaderos.

In [ ]:
np.random.seed(42)

N_CLUSTERS = 6
N_PER_CLUSTER = 180
N_OUTLIERS = 60
BOUNDS = (-10, 10)

# Mismo tamaño, distinta densidad (desviación estándar)
cluster_stds = [0.35, 0.55, 0.75, 0.95, 0.70, 0.85]

# Centros elegidos para mantener los datos dentro de [-10, 10]
# Los dos últimos centros están muy cerca para forzar solapamiento parcial.
cluster_centers = [
    (-6.0, -5.5),
    (-5.5, 6.0),
    (0.0, 0.0),
    (6.0, -4.0),
    (2.0, 2.2),   # par superpuesto
    (3.4, 2.8),   # par superpuesto
]

X_parts = []
y_parts = []

for cluster_id, (cx, cy) in enumerate(cluster_centers):
    std = cluster_stds[cluster_id]
    points = np.random.normal(loc=(cx, cy), scale=std, size=(N_PER_CLUSTER, 2))
    X_parts.append(points)
    y_parts.append(np.full(N_PER_CLUSTER, cluster_id))

X_inliers = np.vstack(X_parts)
y_inliers = np.concatenate(y_parts)

# Outliers uniformes en todo el cuadrado [-10, 10]
X_outliers = np.random.uniform(BOUNDS[0], BOUNDS[1], size=(N_OUTLIERS, 2))
y_outliers = np.full(N_OUTLIERS, -1)

X_all = np.vstack([X_inliers, X_outliers])
y_all = np.concatenate([y_inliers, y_outliers])

print(f"Inliers: {len(X_inliers)} puntos ({N_CLUSTERS} grupos x {N_PER_CLUSTER})")
print(f"Outliers: {len(X_outliers)} puntos")
print(f"Rango X: [{X_all[:, 0].min():.2f}, {X_all[:, 0].max():.2f}]")
print(f"Rango Y: [{X_all[:, 1].min():.2f}, {X_all[:, 1].max():.2f}]")
print("Desviaciones estándar por grupo:", cluster_stds)

### Visualización con etiquetas verdaderas (ground truth)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

inlier_mask = y_all >= 0
outlier_mask = y_all < 0

scatter_inliers = ax.scatter(
    X_all[inlier_mask, 0],
    X_all[inlier_mask, 1],
    c=y_all[inlier_mask],
    cmap="tab10",
    s=18,
    alpha=0.75,
    label="Grupos gaussianos",
)
ax.scatter(
    X_all[outlier_mask, 0],
    X_all[outlier_mask, 1],
    c="black",
    marker="x",
    s=35,
    label="Outliers uniformes",
)

for idx, (cx, cy) in enumerate(cluster_centers):
    ax.scatter(cx, cy, c="red", marker="*", s=180, edgecolors="k")
    ax.text(cx + 0.15, cy + 0.15, f"G{idx}", fontsize=9, fontweight="bold")

ax.set_xlim(BOUNDS)
ax.set_ylim(BOUNDS)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Datos sintéticos: 6 grupos + outliers")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
plt.show()

## 3. Remoción de outliers con Local Outlier Factor (LOF)

K-Means es sensible a valores atípicos porque estos desplazan los centroides. Se detectan outliers con **LOF** y se conserva únicamente el conjunto de inliers para el clustering.

Siguiendo el cuaderno de referencia, los puntos con puntaje LOF alto se marcan como atípicos y se eliminan antes de evaluar distintos valores de **K**.

In [ ]:
contamination = N_OUTLIERS / len(X_all)
lof = LocalOutlierFactor(n_neighbors=35, contamination=contamination)
lof_labels = lof.fit_predict(X_all)
lof_scores = lof.negative_outlier_factor_

inlier_mask_lof = lof_labels == 1
X_clean = X_all[inlier_mask_lof]
y_clean = y_all[inlier_mask_lof]

print(f"Puntos originales: {len(X_all)}")
print(f"Puntos tras remover outliers: {len(X_clean)}")
print(f"Outliers detectados por LOF: {(~inlier_mask_lof).sum()}")

radius = (lof_scores.max() - lof_scores) / (lof_scores.max() - lof_scores.min())
is_outlier_plot = radius > 0.10

plt.figure(figsize=(8, 7))
plt.scatter(X_all[:, 0], X_all[:, 1], color="k", s=8, alpha=0.35, label="Todos los puntos")
plt.scatter(
    X_all[is_outlier_plot, 0],
    X_all[is_outlier_plot, 1],
    s=900 * radius[is_outlier_plot],
    facecolors="none",
    edgecolors="r",
    label="Outliers LOF",
)
plt.xlim(BOUNDS)
plt.ylim(BOUNDS)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Detección de outliers con LOF")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 4. K-Means sobre datos limpios

Con los outliers removidos, se aplica K-Means. Cada punto se asigna al cluster cuyo **centroide** está más cerca (distancia euclidiana). Los atributos `labels_` y `cluster_centers_` contienen, respectivamente, la etiqueta asignada y la posición de cada centroide.

In [ ]:
K_TRUE = N_CLUSTERS
kmeans_demo = KMeans(n_clusters=K_TRUE, random_state=0, n_init=10)
labels_demo = kmeans_demo.fit_predict(X_clean)
centroids_demo = kmeans_demo.cluster_centers_

plt.figure(figsize=(8, 7))
plt.scatter(X_clean[:, 0], X_clean[:, 1], c=labels_demo, cmap="viridis", s=18, alpha=0.8)
plt.scatter(
    centroids_demo[:, 0],
    centroids_demo[:, 1],
    c="red",
    marker="x",
    s=200,
    linewidths=2,
    label="Centroides",
)
plt.xlim(BOUNDS)
plt.ylim(BOUNDS)
plt.xlabel("X")
plt.ylabel("Y")
plt.title(f"K-Means con K = {K_TRUE} (datos sin outliers)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 5. Selección del número óptimo de grupos (K = 2 … 12)

Se evalúan tres criterios:

| Método | Idea principal | K óptimo |
|--------|----------------|----------|
| **Codo** | Busca donde la inercia deja de disminuir rápidamente | Punto de mayor curvatura en la curva inercia vs K |
| **Silhouette** | Mide separación intra/inter cluster | K que maximiza el coeficiente promedio |
| **Ground truth (ARI)** | Compara clustering vs etiquetas reales | K que maximiza el Adjusted Rand Index |

In [ ]:
K_RANGE = range(2, 13)
RANDOM_STATE = 0
N_INIT = 10

inertias = []
silhouette_scores = []
ari_scores = []

for k in K_RANGE:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    pred = model.fit_predict(X_clean)

    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_clean, pred))
    ari_scores.append(adjusted_rand_score(y_clean, pred))

k_list = list(K_RANGE)


def find_elbow_k(k_values, values):
    """Encuentra el codo como el punto más alejado de la recta entre extremos."""
    k_arr = np.array(k_values, dtype=float)
    v_arr = np.array(values, dtype=float)
    p1 = np.array([k_arr[0], v_arr[0]])
    p2 = np.array([k_arr[-1], v_arr[-1]])
    line_vec = p2 - p1
    line_len = np.linalg.norm(line_vec)
    if line_len == 0:
        return k_values[0]
    distances = []
    for k, v in zip(k_arr, v_arr):
        p = np.array([k, v])
        # Distancia perpendicular de p a la recta p1-p2 (evita cross 2D deprecado).
        dist = np.abs(line_vec[0] * (p1[1] - p[1]) - line_vec[1] * (p1[0] - p[0])) / line_len
        distances.append(dist)
    return k_values[int(np.argmax(distances))]


k_elbow = find_elbow_k(k_list, inertias)
k_silhouette = k_list[int(np.argmax(silhouette_scores))]
k_ground_truth = k_list[int(np.argmax(ari_scores))]

print("Resultados de selección de K:")
print(f"  Método del codo (inercia):     K* = {k_elbow}")
print(f"  Método Silhouette:             K* = {k_silhouette}")
print(f"  Método ground truth (ARI max): K* = {k_ground_truth}")
print(f"  K verdadero del dataset:       K  = {K_TRUE}")

### 5.1 Método del codo

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_list, inertias, marker="o")
plt.axvline(k_elbow, color="red", linestyle="--", label=f"K* = {k_elbow}")
plt.xlabel("Número de clusters (K)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### 5.2 Método Silhouette

Para cada punto, el coeficiente de Silhouette compara qué tan cerca está de su propio cluster frente al cluster vecino más cercano. Se elige el **K** que maximiza el promedio global.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_list, silhouette_scores, marker="o")
plt.axvline(k_silhouette, color="red", linestyle="--", label=f"K* = {k_silhouette}")
plt.xlabel("Número de clusters (K)")
plt.ylabel("Coeficiente de Silhouette")
plt.title("Selección de K por Silhouette")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
for n_clusters in [k_silhouette, K_TRUE]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    clusterer = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=N_INIT)
    cluster_labels = clusterer.fit_predict(X_clean)
    silhouette_avg = silhouette_score(X_clean, cluster_labels)
    sample_silhouette_values = silhouette_samples(X_clean, cluster_labels)

    ax1.set_xlim([-0.1, 1])
    ax1.set_ylim([0, len(X_clean) + (n_clusters + 1) * 10])

    y_lower = 10
    for i in range(n_clusters):
        ith_cluster = sample_silhouette_values[cluster_labels == i]
        ith_cluster.sort()
        size_cluster_i = ith_cluster.shape[0]
        y_upper = y_lower + size_cluster_i
        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")
    ax1.set_title("Silhouette por cluster")
    ax1.set_xlabel("Coeficiente de Silhouette")
    ax1.set_ylabel("Etiqueta de cluster")

    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(X_clean[:, 0], X_clean[:, 1], marker=".", s=25, c=colors, edgecolor="k", alpha=0.75)
    centers = clusterer.cluster_centers_
    ax2.scatter(centers[:, 0], centers[:, 1], marker="o", c="white", s=180, edgecolors="k")
    for i, c in enumerate(centers):
        ax2.scatter(c[0], c[1], marker=f"${i}$", s=50, edgecolors="k")
    ax2.set_xlim(BOUNDS)
    ax2.set_ylim(BOUNDS)
    ax2.set_title("Visualización del clustering")
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")

    fig.suptitle(
        f"Análisis de Silhouette con K = {n_clusters} (promedio = {silhouette_avg:.3f})",
        fontsize=13,
        fontweight="bold",
    )
    plt.show()

### 5.3 Método basado en ground truth (Adjusted Rand Index)

Como conocemos las etiquetas verdaderas de los inliers, podemos medir qué tan bien coincide cada partición de K-Means con la verdad operativa. El **ARI** penaliza asignaciones aleatorias y toma valores cercanos a 1 cuando el agrupamiento es casi perfecto.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_list, ari_scores, marker="o")
plt.axvline(k_ground_truth, color="red", linestyle="--", label=f"K* = {k_ground_truth}")
plt.axvline(K_TRUE, color="green", linestyle=":", label=f"K verdadero = {K_TRUE}")
plt.xlabel("Número de clusters (K)")
plt.ylabel("Adjusted Rand Index (ARI)")
plt.title("Selección de K usando ground truth")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

for k in [k_ground_truth, K_TRUE]:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    pred = model.fit_predict(X_clean)
    print(f"K = {k:2d} -> ARI = {adjusted_rand_score(y_clean, pred):.4f}")

## 6. Comparación final de métodos

La tabla resume el **K** recomendado por cada criterio. En la práctica:

- **Silhouette** y **ARI** suelen ser más informativos cuando hay solapamiento parcial entre grupos.
- El **codo** es rápido e intuitivo, pero puede ser ambiguo cuando la curva no presenta un quiebre claro.
- El método con **ground truth** solo está disponible en experimentos controlados; en datos reales se usan Silhouette, codo u otras heurísticas.

In [ ]:
comparison = [
    ("Codo (inercia)", k_elbow),
    ("Silhouette", k_silhouette),
    ("Ground truth (ARI)", k_ground_truth),
    ("K verdadero del dataset", K_TRUE),
]

print(f"{'Método':<28} {'K sugerido':>12}")
print("-" * 42)
for method, k_value in comparison:
    marker = "  <-- coincide" if k_value == K_TRUE else ""
    print(f"{method:<28} {k_value:>12}{marker}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = [
    ("Codo", k_elbow, inertias, "Inercia"),
    ("Silhouette", k_silhouette, silhouette_scores, "Silhouette"),
    ("Ground truth", k_ground_truth, ari_scores, "ARI"),
]

for ax, (name, k_opt, values, ylabel) in zip(axes, methods):
    ax.plot(k_list, values, marker="o")
    ax.axvline(k_opt, color="red", linestyle="--", label=f"K* = {k_opt}")
    ax.axvline(K_TRUE, color="green", linestyle=":", label=f"K real = {K_TRUE}")
    ax.set_title(name)
    ax.set_xlabel("K")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

plt.suptitle("Comparación de métodos para seleccionar K", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Conclusiones

1. Se generó un dataset 2D con **6 clusters gaussianos isotrópicos** de igual cardinalidad y distinta desviación estándar, incluyendo **dos grupos parcialmente superpuestos**.
2. Se incorporaron **outliers uniformes** en `[-10, 10]` y se eliminaron con **LOF** antes de clusterizar.
3. Se exploró **K ∈ {2, …, 12}** usando **codo**, **Silhouette** y **ARI** con etiquetas verdaderas.
4. Cuando dos grupos se solapan, **Silhouette** y el **codo** pueden subestimar **K** (por ejemplo, sugiriendo 4 grupos bien separados), mientras que el **ARI** con ground truth recupera el valor real (**K = 6**). Por eso conviene usar varios criterios y, cuando existan etiquetas, validar con métricas externas.